# Exploration of Dam Removal Dataset

In [1]:
import pandas as pd

## Initial investigation

In [2]:
dam_df = pd.read_csv('../data/raw/dams/ARDamRemovalList_Figshare_Mar2026.csv')
dam_df.head()

,AR_ID,NABI_ID,NID_ID,Dam_Name,Year_Removed,Latitude,Longitude,City_County,River,HUC8,...,Dam_Length_ft,Year_Built,Owner,Original_Use,Type_Material,Reason_for_removal,Miles_Restored_US,Miles_Restored_DS,River_Miles_Reported,Last_Updated
0,AK-001,AK91779,NaN,Lake Bettinger Complex Dam,NaN,57.80333,-152.395,Kodiak,NaN,Kodiak-Afognak Islands,...,251,1959,City of Kodiak,Water Supply,Timber,Delapidated,NaN,NaN,NaN,2/10/2026
1,AK-002,AK91780,NaN,Bettinger Lower Reservoir Dam B,NaN,57.80167,-152.395,Kodiak,NaN,Kodiak-Afognak Islands,...,245,1959,City of Kodiak,Water Supply,Timber,Safety,NaN,NaN,NaN,2/10/2026
2,AK-003,AK91781,NaN,Memory Estates Dam #1,NaN,62.61833,-149.427,Matanuska-Susitna,NaN,Talkeetna River,...,379,1973,NaN,Recreation,Earthen,NaN,NaN,NaN,NaN,2/10/2026
3,AK-004,AK91782,NaN,Memory Estates Dam #2,NaN,62.62667,-149.380,Matanuska-Susitna,NaN,Talkeetna River,...,444,1973,NaN,Recreation,Earthen,NaN,NaN,NaN,NaN,2/10/2026
4,AK-005,NaN,NaN,Switzer One Dam,1988,NaN,NaN,"Tongass National Forest, Juneau",Switzer Creek,Lynn Canal,...,NaN,NaN,NaN,NaN,NaN,Safety,NaN,NaN,NaN,11/10/2017


In [3]:
dam_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2325 entries, 0 to 2324
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   AR_ID                 2325 non-null   object 
 1   NABI_ID               2122 non-null   object 
 2   NID_ID                537 non-null    object 
 3   Dam_Name              2325 non-null   object 
 4   Year_Removed          2185 non-null   object 
 5   Latitude              2126 non-null   float64
 6   Longitude             2126 non-null   float64
 7   City_County           2167 non-null   object 
 8   River                 2252 non-null   object 
 9   HUC8                  2140 non-null   object 
 10  State                 2325 non-null   object 
 11  Dam_Height_ft         1766 non-null   object 
 12  Dam_Length_ft         1376 non-null   object 
 13  Year_Built            1118 non-null   object 
 14  Owner                 1158 non-null   object 
 15  Original_Use         

In [12]:
dam_df.dtypes

AR_ID                    object
NABI_ID                  object
NID_ID                   object
Dam_Name                 object
Year_Removed             object
Latitude                float64
Longitude               float64
City_County              object
River                    object
HUC8                     object
State                    object
Dam_Height_ft            object
Dam_Length_ft            object
Year_Built               object
Owner                    object
Original_Use             object
Type_Material            object
Reason_for_removal       object
Miles_Restored_US        object
Miles_Restored_DS        object
River_Miles_Reported     object
Last_Updated             object
dtype: object

Except for latitude and longitude, all features are object types. In order to get informative descriptive statistics and perform more extensive EDA, some of these features will be converted to numeric data types below.

In [25]:
numeric_cols = [
    "Dam_Height_ft",
    "Dam_Length_ft",
    "Miles_Restored_US",
    "Miles_Restored_DS",
    "River_Miles_Reported"
]

for col in numeric_cols:
    dam_df[col] = pd.to_numeric(dam_df[col], errors="coerce")

dam_df["Year_Removed"] = pd.to_numeric(
    dam_df["Year_Removed"], errors="coerce"
).astype("Int64")

dam_df["Year_Built"] = pd.to_numeric(
    dam_df["Year_Built"], errors="coerce"
).astype("Int64")

dam_df["Last_Updated"] = pd.to_datetime(
    dam_df["Last_Updated"], errors="coerce"
)

text_cols = [
    "AR_ID", "NABI_ID", "NID_ID", "Dam_Name",
    "City_County", "River", "HUC8", "State",
    "Owner", "Original_Use", "Type_Material",
    "Reason_for_removal"
]

dam_df[text_cols] = dam_df[text_cols].astype("string")

In [26]:
dam_df.describe()

,Year_Removed,Latitude,Longitude,Dam_Height_ft,Dam_Length_ft,Year_Built,Miles_Restored_US,Miles_Restored_DS,River_Miles_Reported,Last_Updated
count,2182.0,2126.000000,2126.000000,1730.00000,1355.000000,786.0,453.000000,277.000000,486.000000,2325
mean,2009.992667,40.867210,-88.668649,14.33078,231.809668,1924.189567,33.530221,101.715632,21.774835,2025-06-26 23:40:48
min,1912.0,18.455111,-159.673562,1.00000,2.000000,1708.0,0.000000,0.000000,0.000000,2017-11-10 00:00:00
25%,2005.0,39.766655,-93.897923,5.26250,50.000000,1906.0,1.000000,1.600000,1.100000,2026-02-10 00:00:00
50%,2013.0,41.249593,-82.348913,10.00000,120.000000,1931.0,4.000000,4.000000,4.000000,2026-02-10 00:00:00
75%,2019.0,43.469702,-75.658200,17.00000,260.000000,1957.0,16.000000,16.000000,17.000000,2026-02-10 00:00:00
max,2025.0,65.292800,-66.702972,210.00000,7500.000000,2023.0,2622.000000,7275.000000,1873.000000,2026-03-12 00:00:00
std,14.22186,4.447030,18.227580,15.79745,433.489105,44.306155,152.995813,667.602295,97.207537,NaN


## Potential columns to join on

In [33]:
hucs = dam_df["HUC8"].dropna().unique()
print(f"The dam removal dataset has {len(hucs)} unique HUC8 watersheds")

The dam removal dataset has 678 unique HUC8 watersheds
